# Part 5 — Historical Test and Evaluation

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Evaluate the selected development specification and independently thresholded prior baseline once. All tables and figures are reconstructed from sealed prediction evidence. No estimator or alarm threshold is fitted here.

## 1. Repository and runtime
Use a fresh CPU runtime, Java 17 or later, and the pinned dependencies. Use the same Python and dependency versions for both Part 5 notebooks.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
sys.path.insert(0,str(REPO/'src'))
def command(action):
    proc = subprocess.Popen([sys.executable,str(REPO/'run_part5.py'),action,'--workspace',str(WORKSPACE)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='',flush=True)
    if proc.wait()!=0:
        raise RuntimeError(f'Part 5 {action} failed; inspect diagnostics')

## 2. Mandatory offline validation
Run the complete test suite, including real Parquet and Spark integration. A failed or skipped test stops the pipeline.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path('/content/part5_tests.xml')
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,'-m','pytest','-q','-ra','--tb=short','--show-capture=no',f'--junitxml={result_path}','tests'],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',SPARK_LOCAL_IP='127.0.0.1',PYSPARK_PYTHON=sys.executable,MPLBACKEND='Agg'),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError('Missing pytest report')
cases=list(ET.parse(result_path).getroot().iter('testcase'))
problems=[(case,node) for case in cases for node in case if node.tag in ('failure','error','skipped')]
for case,node in problems:
    print(case.get('classname'),case.get('name'),node.tag,node.get('message'),node.text or '')
if result.returncode or not cases or problems: raise RuntimeError('Offline validation failed; inspect diagnostics')
TESTS_PASSED=True
print(f'Mandatory offline validation: PASS ({len(cases)} tests, zero skips)')

## 3. Verified upstream identities
Check the sealed Gold and Silver inventories and Bronze manifest identity. Development does not decode the historical test partitions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORKSPACE=Path('/content/drive/MyDrive/11230945')
assert TESTS_PASSED
from fdic_ews.modelling.pipeline import upstream, evidence_path, verify_development, verify_oof
from fdic_ews.contracts import sha256
GOLD, gm, SILVER, IDENTITY = upstream(WORKSPACE)
EVIDENCE=evidence_path(WORKSPACE,IDENTITY)
ATTEMPT=EVIDENCE.parent/'test_attempt.json'
print('Verified upstream identities:',json.dumps(IDENTITY,indent=2))
print('Gold decision quarters:',gm['summary']['decision_quarters'])
print('Gold predictors:',gm['summary']['feature_count'])
print('Represented test-calendar failures:',gm['summary']['test_distinct_failure_certs_in_calendar_period'])
print('Evidence directory:',EVIDENCE)

## 4. Confirm the sealed development protocol
The Python and package versions must match the sealed environment before the first test attempt. Read the model, features, cost policy and thresholds; do not modify them.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from fdic_ews.modelling.pipeline import environment
DM=verify_development(EVIDENCE,IDENTITY)
S=DM['summary']; SELECTED=S['selected_model']
print('Selection:',SELECTED)
print('Locked thresholds:',S['thresholds'])
print('Cost policy:',json.dumps(S['cost_policy'],indent=2))
print('Development seal:',sha256(EVIDENCE/'development_manifest.json'))
print('Recorded runtime:',json.dumps(DM['environment'],indent=2))
if not ATTEMPT.exists():
    assert environment()==DM['environment'], 'Runtime mismatch: resolve before authorizing the test' 

## 5. Authorize the single historical evaluation
After checking Notebook 06, change `AUTHORIZE_HISTORICAL_TEST` to `True` for the first evaluation only. A workspace-wide marker is created before the test is decoded or scored. If a completed test exists, the cell verifies saved predictions instead. If an attempted test is incomplete, stop and document the failure; do not remove the marker.

In [ ]:
AUTHORIZE_HISTORICAL_TEST = False  # Change to True once after inspecting the seal.
TEST=EVIDENCE/'test'
if ATTEMPT.exists():
    if not (TEST/'manifest.json').is_file():
        raise RuntimeError('Historical test was attempted but is incomplete. Preserve the marker and evidence; do not rescore.')
    print('Existing test evidence: read-only verification.')
else:
    if not AUTHORIZE_HISTORICAL_TEST:
        raise RuntimeError('Inspect the sealed development result, then explicitly authorize the single evaluation.')
    command('evaluate')
command('verify')
TM=json.loads((TEST/'manifest.json').read_text())
assert TM['state']=='VERIFIED'
print('Test state:',TM['state'])
print('Test seal SHA-256:',sha256(TEST/'manifest.json'))

## 6. Historical test ranking and operating performance
Report AP, ROC-AUC, Brier score, log loss and quarter-wise top-5% recall separately from operating metrics at the frozen threshold. The baseline has its own development-optimized threshold. No-alarm and all-alarm reference policies are computed without refitting.

In [ ]:
metrics=pd.read_csv(TEST/'test_metrics.csv'); refs=pd.read_csv(TEST/'reference_policies.csv')
print('Table 1. Ranking and probability diagnostics'); display(metrics[['model','rows','positives','prevalence','ap','roc_auc','brier','log_loss','recall_top5pct_by_quarter']].round(6))
print('Table 2. Frozen operating decisions'); display(metrics[['model','threshold','tp','fp','fn','tn','precision','recall','fpr','alert_rate']].round(6))
print('Table 3. Modelled economic loss'); display(metrics[['model','fp','missed_events','loss_usd_thousands']].round(3))
display(refs[['policy','fp','missed_events','loss_usd_thousands']].round(3))
import pyarrow.parquet as pq
predictions={name:pq.ParquetFile(TEST/f'predictions_{name}.parquet').read().to_pandas() for name in [SELECTED,'prior']}
from sklearn.metrics import precision_recall_curve,roc_curve
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig,ax=plt.subplots(figsize=(6.4,3.6))
for name,p in predictions.items():
    precision,recall,_=precision_recall_curve(p.Y_FAIL_4Q,p.score); ax.plot(recall,precision,label=name)
ax.axhline(float(metrics.prevalence.iloc[0]),color='#8c9ca6',linestyle='--',label='Test prevalence'); ax.set(xlabel='Recall',ylabel='Precision',xlim=(0,1),ylim=(0,1)); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.4,3.6))
for name,p in predictions.items():
    fpr,tpr,_=roc_curve(p.Y_FAIL_4Q,p.score); ax.plot(fpr,tpr,label=name)
ax.plot([0,1],[0,1],'--',color='#8c9ca6'); ax.set(xlabel='False-positive rate',ylabel='True-positive rate'); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)

## 7. Event detection and warning lead time
Count distinct CERT–failure-date events once. The lead time is the number of days from the earliest positive alert to the failure; it is conditional on detection. Distinguish failures occurring within the test calendar from the events represented through the final test outcome horizon.

In [ ]:
print('Table 4. Event-level outcomes through the test horizons'); display(metrics[['model','events','detected_events','missed_events','event_recall','median_lead_days','lead_at_least_90_days']].round(4))
print('Table 5. Events within the historical test calendar'); display(pd.read_csv(TEST/'test_calendar_events.csv').round(4))
events=pd.read_csv(TEST/f'events_{SELECTED}.csv')
r=metrics.loc[metrics.model.eq(SELECTED)].iloc[0]; cm=np.array([[r.tn,r.fp],[r.fn,r.tp]],dtype=int)
fig,ax=plt.subplots(figsize=(4.6,3.6)); ax.imshow(np.log1p(cm),cmap='Blues');
for i in range(2):
    for j in range(2): ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',color='white' if np.log1p(cm[i,j])>np.log1p(cm).max()*.65 else 'black')
ax.set(xticks=[0,1],xticklabels=['No alarm','Alarm'],yticks=[0,1],yticklabels=['No failure','Failure'],xlabel='Decision',ylabel='Observed outcome'); fig.tight_layout(); plt.show(); plt.close(fig)
lead=events.loc[events.detected,'lead_days'].dropna()
if not lead.empty:
    fig,ax=plt.subplots(figsize=(6.4,3.2)); ax.hist(lead,bins=np.arange(0,400,30),color='#365c72',edgecolor='white'); ax.set(xlabel='Days from earliest correct alert to failure',ylabel='Detected events'); fig.tight_layout(); plt.show(); plt.close(fig)
else: print('No detected events; lead time is undefined.')

## 8. Calibration and regime diagnostics
Reliability bins are descriptive; no calibration is fitted on the test. Annual decision-period metrics describe temporal heterogeneity. Events can be represented in multiple decision years, so annual event counts and economic losses are not additive.

In [ ]:
annual=pd.read_csv(TEST/'annual_metrics.csv')
print('Table 6. Annual decision-period diagnostics'); display(annual[['model','year','rows','positives','prevalence','ap','roc_auc','recall','event_recall','fp','missed_events']].round(6))
fig,ax=plt.subplots(figsize=(6.5,3.5))
for name in [SELECTED,'prior']:
    cal=pd.read_csv(TEST/f'calibration_{name}.csv'); ax.plot(cal.mean_score,cal.observed_rate,marker='o',label=name)
ax.plot([0,1],[0,1],'--',color='#8c9ca6'); ax.set(xlabel='Mean predicted score in bin',ylabel='Observed positive rate'); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.5,3.3))
for name,g in annual.groupby('model'):
    ax.plot(g.year,g.ap,marker='o',label=name)
ax.set(xlabel='Decision year',ylabel='Average precision'); ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show(); plt.close(fig)

## 9. Verification and conclusion handoff
The test is closed. Reopening the notebook must only verify saved evidence. Interpretation is limited to the fixed historical period, adjudicable outcomes, assumed information availability and proxy economic costs. Save this notebook with its outputs; do not revise model choices from test results.

In [ ]:
print('Development seal:',sha256(EVIDENCE/'development_manifest.json'))
print('Workspace attempt marker:',ATTEMPT)
print('Test seal:',sha256(TEST/'manifest.json'))
print('Test rows:',TM['rows'])
print('Selected model:',SELECTED)
print('Status: VERIFIED — saved prediction evidence; no further test scoring')